### Objective

This notebook focus ok building and testing all parts of the Ghbot (Grid Hedge Bot).

In [1]:
# Custom Notebook Settings

template = """
[sweep]
mode = "fast"

[params]
fast = [9]
slow = [21]

[botrun.runtime]
mode = "sweep"
max_loop = 0

[botrun.market]
symbol = "BTC"
interval = "1m"

[botrun.backtest]
start = "2025-01-01"
stop = "2025-01-31"

[[botrun.signalers]]
name = "emacross"
interval = "1m"

[botrun.signalers.params]
fast = 9
slow = 21

[botrun.executor]
name = "tradebot"
take_profit_pct = 2.0
stop_loss_pct = 1.0
max_cycles = 0

[botrun.risk]
score = 1
"""

# 0 creates a new sweep. Nonzero loads and reruns that sweep_id.
SWEEP_ID = 0
FEE_PCT = 0.5   # 0.5%

In [2]:
# Imports

import json
import tomllib

from nuubot import Nuubot
from nuubot.core.models.mconfig import SweepConfig
from nuubot.datastore import SweepRow


# Standard Setup

nuubot = Nuubot().setup()

In [3]:
# Create the Sweep Record

def insert_sweep_record(template: str) -> int:
    template_data = tomllib.loads(template)
    template_data["botrun"]["runtime"].setdefault("loop_seconds", 1.0)
    template_data["botrun"]["backtest"]["data_dir"] = f"{nuubot.config.paths.data_dir}/binance/raw/spot/monthly/klines"
    SweepConfig.model_validate(template_data)
    config_json = json.dumps(template_data, sort_keys=True, separators=(",", ":"))

    with nuubot.datastore.session(nuubot.config.databases.sweeps) as session:
        sweep = SweepRow(
            sweep_desc="ghbot_notebook_smoke",
            config_json=config_json,
            results_json="{}",
            status="configured",
            sweeprun_count=1,
        )
        session.add(sweep)
        session.commit()
        return sweep.sweep_id

sweep_id = insert_sweep_record(template) if SWEEP_ID == 0 else SWEEP_ID
print(f"Sweep ID is {sweep_id}")

Sweep ID is 1
